# Problem 2: Clan Rank Regression - EDA

## Diagnostic and comparison of full and trophy-free dataset variants

This notebook analyzes the dataset for **Problem 2: Clan Rank Regression**. It evaluates two variants: the full dataset including direct trophy metrics, and a trophy-free variant built solely from clan composition, activity, and progression signals.

## 1. Setup and Dynamic Data Loading

Libraries are imported and plotting styles are configured. The project root is located dynamically, the dataset is loaded, and two DataFrame views are created: `df_full` and `df_no_trophies`.

In [ ]:
# === 1. Setup and Dynamic Data Loading ===
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.feature_selection import mutual_info_regression
from pathlib import Path
from collections import defaultdict

sns.set_theme(style='whitegrid', palette='viridis')
%matplotlib inline

def find_project_root() -> Path:
    """Locate the project root by searching for the Problem 2 dataset."""
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for cand in candidates:
        if (cand / 'data' / 'datasets' / 'clan_rank_regression.parquet').exists():
            return cand
    raise FileNotFoundError(
        'Could not find data/datasets/clan_rank_regression.parquet. '
        'Adjust the path or run the notebook from the project root.'
    )

root = find_project_root()
DATA_PATH = root / 'data' / 'datasets' / 'clan_rank_regression.parquet'
print(f'Project root directory: {root}')
print(f'Dataset path: {DATA_PATH}')

# Load the dataset
df = pd.read_parquet(DATA_PATH)
print(f'Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.')

# Create the two analysis variants
TARGET = 'clan_rank'
trophy_cols = [c for c in df.columns if any(k in c.lower() for k in ['troph', 'points', 'versus'])]
print('')
print(f'Direct trophy-related columns excluded: {trophy_cols}')

df_full = df.copy()
df_no_trophies = df.drop(columns=trophy_cols)

print('')
print(f'Full variant shape: {df_full.shape}')
print(f'Trophy-free variant shape: {df_no_trophies.shape}')
df_full.head()

## 2. Data Integrity and Quality Audit

Dimensions, data types, missing values, duplicate rows, and quasi-constant features are audited for both variants.

In [ ]:
# === 2. Data Integrity and Quality Audit ===

def audit_dataframe(data: pd.DataFrame, name: str) -> None:
    """Prints basic integrity metrics for a DataFrame."""
    print('')
    print(f'=== {name} ===')
    print(f'Dimensions: {data.shape[0]} rows x {data.shape[1]} columns')
    print('')
    print('Data types:')
    display(data.dtypes.to_frame(name='dtype'))
    missing = data.isna().sum()
    missing_pct = (missing / len(data)) * 100
    missing_df = pd.DataFrame({'missing': missing, 'percentage': missing_pct})
    print('')
    print('Missing values (columns with missing > 0):')
    display(missing_df[missing_df['missing'] > 0].sort_values('percentage', ascending=False))
    dup_rows = data.duplicated().sum()
    print('')
    print(f'Exact duplicate rows: {dup_rows}')
    threshold = 0.95
    constant_features = []
    for col in data.columns:
        value_counts = data[col].value_counts(dropna=False)
        top_freq = value_counts.iloc[0] / len(data) if len(value_counts) > 0 else 1.0
        if top_freq >= threshold:
            constant_features.append((col, 'nearly_constant', top_freq))
        if pd.api.types.is_numeric_dtype(data[col]) and data[col].nunique(dropna=True) <= 1:
            constant_features.append((col, 'zero_variance', top_freq))
    if constant_features:
        print('')
        print(f'Quasi-constant or zero-variance features (threshold={threshold}):')
        display(pd.DataFrame(constant_features, columns=['column', 'type', 'mode_frequency']))
    else:
        print('')
        print(f'No quasi-constant features with mode frequency >= {threshold}.')

audit_dataframe(df_full, 'Full Dataset')
audit_dataframe(df_no_trophies, 'Trophy-Free Dataset')

## 3. Target Analysis (`clan_rank`)

Descriptive statistics, distribution plots in raw and log-transformed scale, and skewness diagnostics are performed for the target variable.

In [ ]:
# === 3. Target Analysis (clan_rank) ===

target = df[TARGET].copy()
print('=== Descriptive statistics for clan_rank ===')
print(target.describe().to_string())
print('')
skewness = stats.skew(target.dropna())
kurtosis = stats.kurtosis(target.dropna())
print(f'Skewness: {skewness:.4f}')
print(f'Kurtosis: {kurtosis:.4f}')

# Raw scale distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(target, kde=True, ax=axes[0])
axes[0].set_title('clan_rank distribution (raw scale)')
axes[0].set_xlabel('clan_rank')
axes[0].set_ylabel('Frequency')

# Log-transformed scale
log_target = np.log1p(target)
sns.histplot(log_target, kde=True, ax=axes[1])
axes[1].set_title('clan_rank distribution (log1p scale)')
axes[1].set_xlabel('log1p(clan_rank)')
axes[1].set_ylabel('Frequency')
plt.tight_layout()
plt.show()

# Log-transform recommendation
if abs(skewness) > 1.0:
    print('The clan_rank distribution is highly skewed. A log1p transformation is recommended before modeling.')
elif abs(skewness) > 0.5:
    print('The clan_rank distribution is moderately skewed. A log1p transformation may improve model performance.')
else:
    print('The clan_rank distribution is approximately symmetric. No transformation is strictly required.')

## 4. Feature Group Profiling

Variables are classified into conceptual groups: **Progression**, **Activity**, **Economy**, **Trophies/Competition**, and **Clan Composition**. Skewness, zero-value ratio, and missing percentage are reported for each numerical feature.

In [ ]:
# === 4. Feature Group Profiling ===

def categorize_feature(col: str) -> str:
    """Assigns a feature to a conceptual group based on its name."""
    col_l = col.lower()
    if any(k in col_l for k in ['troop', 'hero', 'spell', 'equipment', 'builder_hall', 'town_hall', 'exp_level', 'achievement']):
        return 'Progression'
    if any(k in col_l for k in ['donat', 'attack', 'defense', 'war_stars', 'versus_battle']):
        return 'Activity'
    if any(k in col_l for k in ['loot', 'gold', 'elixir', 'dark_elixir', 'clan_games']):
        return 'Economy'
    if any(k in col_l for k in ['troph', 'points', 'league', 'legend']):
        return 'Trophies/Competition'
    if any(k in col_l for k in ['members', 'clan_', 'count', 'composition', 'size']):
        return 'Clan Composition'
    return 'Other'

def profile_numeric_features(data: pd.DataFrame, name: str) -> pd.DataFrame:
    """Computes skewness, zero ratio and missing ratio for numeric columns."""
    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    if TARGET in numeric_cols:
        numeric_cols.remove(TARGET)
    records = []
    for col in numeric_cols:
        series = data[col]
        records.append({
            'feature': col,
            'group': categorize_feature(col),
            'skewness': series.skew(),
            'zero_pct': (series == 0).mean() * 100,
            'missing_pct': series.isna().mean() * 100
        })
    profile = pd.DataFrame(records).sort_values('skewness', key=lambda s: s.abs(), ascending=False)
    print('')
    print(f'=== Numerical feature profile for {name} ===')
    display(profile)
    return profile

profile_full = profile_numeric_features(df_full, 'Full Dataset')
profile_no_trophies = profile_numeric_features(df_no_trophies, 'Trophy-Free Dataset')

## 5. Feature-Target Relationships (Regression)

For each variant, Mutual Information, Pearson correlation, and Spearman correlation with `clan_rank` are computed. Scatter plots are shown for the top 6 features by Mutual Information.

In [ ]:
# === 5. Feature-Target Relationships (Regression) ===

def compute_feature_target_metrics(data: pd.DataFrame, name: str) -> dict:
    """Computes MI, Pearson and Spearman correlations for numeric features vs target."""
    target = data[TARGET]
    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    if TARGET in numeric_cols:
        numeric_cols.remove(TARGET)

    # Impute median temporarily for Mutual Information
    X = data[numeric_cols].copy()
    for col in numeric_cols:
        if X[col].isna().any():
            X[col] = X[col].fillna(X[col].median())
    mi_scores = mutual_info_regression(X, target, random_state=42)
    mi_df = pd.DataFrame({'feature': numeric_cols, 'mutual_info': mi_scores})
    mi_df = mi_df.sort_values('mutual_info', ascending=False).reset_index(drop=True)

    pearson = []
    spearman = []
    for col in numeric_cols:
        pearson_r = data[col].corr(target, method='pearson')
        spearman_r = data[col].corr(target, method='spearman')
        pearson.append((col, pearson_r))
        spearman.append((col, spearman_r))
    pearson_df = pd.DataFrame(pearson, columns=['feature', 'pearson_r']).sort_values('pearson_r', key=lambda s: s.abs(), ascending=False)
    spearman_df = pd.DataFrame(spearman, columns=['feature', 'spearman_r']).sort_values('spearman_r', key=lambda s: s.abs(), ascending=False)

    print('')
    print(f'=== Feature-target metrics for {name} ===')
    print('Top 10 Mutual Information:')
    display(mi_df.head(10))
    print('')
    print('Top 10 Pearson correlation:')
    display(pearson_df.head(10))
    print('')
    print('Top 10 Spearman correlation:')
    display(spearman_df.head(10))

    # Scatter plots for top 6 MI features
    top_features = mi_df.head(6)['feature'].tolist()
    if top_features:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        for ax, var in zip(axes, top_features):
            ax.scatter(data[var], data[TARGET], alpha=0.6)
            ax.set_title(f'{var} vs clan_rank')
            ax.set_xlabel(var)
            ax.set_ylabel(TARGET)
        for ax in axes[len(top_features):]:
            ax.axis('off')
        plt.suptitle(f'Top 6 MI features for {name}')
        plt.tight_layout(rect=[0, 0, 1, 0.97])
        plt.show()

    return {'mi': mi_df, 'pearson': pearson_df, 'spearman': spearman_df}

metrics_full = compute_feature_target_metrics(df_full, 'Full Dataset')
metrics_no_trophies = compute_feature_target_metrics(df_no_trophies, 'Trophy-Free Dataset')

## 6. Multicollinearity and Redundancy

Pearson and Spearman correlation matrices are computed among numerical predictors. Feature pairs exceeding |r| > 0.85 are reported and visual heatmaps are shown.

In [ ]:
# === 6. Multicollinearity and Redundancy ===

def collinearity_report(data: pd.DataFrame, name: str) -> None:
    """Prints high-correlation pairs and displays correlation heatmaps."""
    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    if TARGET in numeric_cols:
        numeric_cols.remove(TARGET)
    corr_data = data[numeric_cols].copy().replace([np.inf, -np.inf], np.nan)
    for col in corr_data.columns:
        if corr_data[col].isna().any():
            corr_data[col] = corr_data[col].fillna(corr_data[col].median())

    pearson_corr = corr_data.corr(method='pearson')
    spearman_corr = corr_data.corr(method='spearman')

    threshold = 0.85

    def get_high_corr_pairs(corr_matrix):
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        pairs = []
        for col in upper.columns:
            for idx in upper.index:
                val = upper.loc[idx, col]
                if pd.notna(val) and abs(val) > threshold:
                    pairs.append((idx, col, val))
        return pd.DataFrame(pairs, columns=['feature_1', 'feature_2', 'corr'])

    pearson_pairs = get_high_corr_pairs(pearson_corr)
    spearman_pairs = get_high_corr_pairs(spearman_corr)

    print('')
    print(f'=== Collinearity report for {name} ===')
    print(f'Pairs with |Pearson| > {threshold}:')
    display(pearson_pairs if len(pearson_pairs) > 0 else None)
    print('')
    print(f'Pairs with |Spearman| > {threshold}:')
    display(spearman_pairs if len(spearman_pairs) > 0 else None)

    plt.figure(figsize=(12, 10))
    sns.heatmap(pearson_corr, annot=False, cmap='coolwarm', center=0, square=True)
    plt.title(f'Pearson correlation matrix - {name}')
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 10))
    sns.heatmap(spearman_corr, annot=False, cmap='coolwarm', center=0, square=True)
    plt.title(f'Spearman correlation matrix - {name}')
    plt.tight_layout()
    plt.show()

collinearity_report(df_full, 'Full Dataset')
collinearity_report(df_no_trophies, 'Trophy-Free Dataset')

## 7. Outlier and Extreme Case Detection

IQR-based outlier percentages are reported for every numerical feature. The top 10 extreme records for influential features are inspected qualitatively.

In [ ]:
# === 7. Outlier and Extreme Case Detection ===

def iqr_outlier_report(data: pd.DataFrame, name: str) -> pd.DataFrame:
    """Computes IQR outlier percentages for all numerical features."""
    numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
    if TARGET in numeric_cols:
        numeric_cols.remove(TARGET)
    records = []
    for col in numeric_cols:
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        mask = (data[col] < lower_bound) | (data[col] > upper_bound)
        records.append({
            'feature': col,
            'q1': q1,
            'q3': q3,
            'IQR': iqr,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'n_outliers': int(mask.sum()),
            'pct_outliers': 100.0 * mask.sum() / len(data)
        })
    report = pd.DataFrame(records).sort_values('n_outliers', ascending=False)
    print('')
    print(f'=== IQR outlier report for {name} ===')
    display(report)
    return report

outliers_full = iqr_outlier_report(df_full, 'Full Dataset')
outliers_no_trophies = iqr_outlier_report(df_no_trophies, 'Trophy-Free Dataset')

# Qualitative inspection of top 10 extreme records for influential features
for variant_name, data in [('Full', df_full), ('Trophy-Free', df_no_trophies)]:
    print('')
    print(f'=== Top 10 extreme records for {variant_name} variant ===')
    for var in ['donations', 'trophies', 'attack_wins']:
        if var in data.columns:
            top_vals = data.nlargest(10, var)[[var, TARGET]].reset_index(drop=True)
            print(f'Extreme values for {var}:')
            display(top_vals)

## 8. Modeling Implications and Executive Summary

The findings are consolidated into an executable summary dictionary and explicit preprocessing, validation, pruning, and modeling recommendations are provided.

In [ ]:
# === 8. Summary dictionary and recommendations ===
target = df[TARGET]
max_pct = (target.value_counts(normalize=True) * 100).max()
min_pct = (target.value_counts(normalize=True) * 100).min()
summary = {
    'target': TARGET,
    'full_rows': df_full.shape[0],
    'full_cols': df_full.shape[1],
    'trophy_free_rows': df_no_trophies.shape[0],
    'trophy_free_cols': df_no_trophies.shape[1],
    'trophy_columns_removed': trophy_cols,
    'target_skewness': float(stats.skew(target.dropna())),
    'target_kurtosis': float(stats.kurtosis(target.dropna())),
    'target_imbalance_ratio': float(max_pct / max(min_pct, 1e-9)) if min_pct > 0 else float('inf'),
    'top_mi_features_full': metrics_full['mi'].head(10).to_dict('records'),
    'top_mi_features_trophy_free': metrics_no_trophies['mi'].head(10).to_dict('records'),
    'full_high_pearson_pairs': collinearity_report_df_full if 'collinearity_report_df_full' in globals() else [],
    'trophy_free_high_pearson_pairs': collinearity_report_df_no_trophies if 'collinearity_report_df_no_trophies' in globals() else [],
    'outlier_max_pct_full': float(outliers_full['pct_outliers'].max()) if len(outliers_full) > 0 else 0.0,
    'outlier_max_pct_trophy_free': float(outliers_no_trophies['pct_outliers'].max()) if len(outliers_no_trophies) > 0 else 0.0
}
print('=== EXECUTIVE SUMMARY EDA P2 ===')
for k, v in summary.items():
    print(f'{k}: {v}')

### Preprocessing Recommendations

- **Imputation**
  - Numeric: median for highly skewed features or when outliers are prevalent; otherwise, mean.
  - Categorical: most frequent category (mode); consider an explicit `"missing"` category if the missing rate is high.
  - Drop columns with more than 40-50% missing values, unless they carry critical domain information.

- **Transformations**
  - Apply `log1p` to severely skewed predictors (`|skew| > 1`) such as donations, loot metrics, or trophy counts.
  - Use `RobustScaler` for features with extreme outliers; `StandardScaler` for the rest.

- **Target Transformation**
  - Based on the skewness diagnostic, transform `clan_rank` with `log1p` if `|skew| > 0.5` and evaluate inverse transform for interpretation.

### Cross-Validation Strategy

- **StratifiedKFold on rank bins**  
  - Bin target values into quantile-based groups and use these bins as stratification labels to preserve distribution in each fold.
- **GroupKFold**  
  - If multiple rows share the same clan entity, use `clan_tag` as the grouping variable to prevent data leakage.
- **Metrics**  
  - `MAE`: scales interpretability in original units.  
  - `RMSE`: penalizes large errors more strongly.  
  - `R²`: proportion of variance explained.

### Feature Pruning

- Remove zero-variance and nearly constant features (`mode frequency >= 0.95`).
- For each collinear pair (`|Pearson| > 0.85` or `|Spearman| > 0.85`), retain the feature with higher Mutual Information.
- If MI is very close, prefer the feature with lower measurement cost or higher interpretability.
- For tree-based models, correlation pruning is less critical, but can reduce training time and improve stability.

### Recommended Baseline Models

1. **Ridge Regression**  
   - `Ridge(alpha=1.0, positive=False)`  
   - Good baseline for regularized linear relationships, especially with high-dimensional data.

2. **Random Forest Regressor**  
   - `RandomForestRegressor(n_estimators=200, random_state=42, max_depth=None)`  
   - Robust to non-linear patterns and outliers; provides feature importance.

3. **XGBoost Regressor**  
   - `XGBRegressor(objective='reg:squarederror', n_estimators=200, learning_rate=0.05, random_state=42)`  
   - Strong performance with hyperparameter tuning and early stopping.

The above recommendations should directly feed the modeling phase for Problem 2.